# Drishti VTOE Worker - Google Colab T4

Runs the in-house Virtual Try-On Engine on NVIDIA T4 (16GB VRAM).

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order
3. Copy the public URL from cell 5

In [ ]:
#@title 1. Check GPU
!nvidia-smi

In [ ]:
#@title 2. Install dependencies (run once, ~3 min)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers peft accelerate xformers
!pip install -q insightface onnxruntime-gpu
!pip install -q controlnet-aux ultralytics
!pip install -q fastapi uvicorn pydantic python-multipart
!pip install -q opencv-python-headless structlog
!pip install -q basicsr facexlib
!pip install -q clean-fid scikit-image
!pip install -q pyngrok nest-asyncio

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('All dependencies installed')

In [ ]:
#@title 3. Clone DRISHTI repo + download CodeFormer weights
!git clone https://github.com/Shuttler14/mynarrative-drishti.git
%cd mynarrative-drishti

!mkdir -p /content/models/codeformer
!wget -q -O /content/models/codeformer/codeformer.pth \
  https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth

print('Repo cloned, CodeFormer weights downloaded')

In [ ]:
#@title 4. Start VTOE server (background)
import subprocess, time, os, signal, sys

# Kill any existing process on port 8001
!fuser -k 8001/tcp 2>/dev/null || true
time.sleep(1)

# Start uvicorn as a subprocess (not in a thread with ! magic)
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'vtoe.api.server:app',
     '--host', '0.0.0.0', '--port', '8001', '--log-level', 'info'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(3)

if server_proc.poll() is None:
    print(f'VTOE server started (PID: {server_proc.pid})')
else:
    out = server_proc.stdout.read().decode()
    print(f'Server failed to start:\n{out}')

In [ ]:
#@title 5. Start Cloudflare Tunnel + get public URL
import subprocess, time, re

# Kill existing cloudflared
!pkill cloudflared 2>/dev/null || true
time.sleep(1)

# Start cloudflared tunnel
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8001'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

# Wait for tunnel URL to appear in output
tunnel_url = None
start = time.time()
while time.time() - start < 30:
    line = tunnel_proc.stdout.readline().decode()
    if not line:
        time.sleep(0.1)
        continue
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    VTOE_URL = tunnel_url
    print(f'\n=== VTOE Worker Ready ===')
    print(f'Public URL: {tunnel_url}')
    print(f'Health: {tunnel_url}/v1/health')
    print(f'Try-on: {tunnel_url}/v1/try-on')
    print(f'Engines: {tunnel_url}/v1/engines')
else:
    print('Tunnel failed to start. Check cloudflared output.')
    VTOE_URL = 'http://localhost:8001'
    print(f'Local URL: {VTOE_URL}')

In [ ]:
#@title 6. Test health + try-on
import requests, base64, io
from PIL import Image

# Health check (use localhost, tunnel is for external access)
LOCAL_URL = 'http://localhost:8001'
try:
    r = requests.get(f'{LOCAL_URL}/v1/health', timeout=10)
    print(f'Health: {r.json()}')
except Exception as e:
    print(f'Health check failed: {e}')

# Test try-on with synthetic images
person = Image.new('RGB', (768, 1024), (200, 180, 160))
garment = Image.new('RGB', (768, 1024), (180, 40, 40))

def img_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()

print('\nSending test try-on request...')
resp = requests.post(f'{LOCAL_URL}/v1/try-on', json={
    'person_image': img_to_b64(person),
    'garment_image': img_to_b64(garment),
    'garment_type': 'ethnic',
    'ethnic_sub_type': 'saree',
    'quality': 'fast',
    'preserve_face': True
}, timeout=120)

if resp.status_code == 200:
    result = resp.json()
    print(f'Status: {result["status"]}')
    print(f'Time: {result["processing_time_ms"]}ms')
    print(f'Quality: {result["quality_score"]}')
    print(f'Face sim: {result["face_similarity"]}')
    print(f'Garment sim: {result["garment_similarity"]}')
    print(f'Model: {result["metadata"]["model_used"]}')
else:
    print(f'Error: {resp.status_code} - {resp.text[:500]}')

In [ ]:
#@title 7. Prepare training data + Train ethnic LoRAs
#@markdown Run this cell to generate synthetic training data and train LoRAs.
#@markdown Each LoRA takes ~45min on T4 (15 epochs, 500 synthetic samples).
#@markdown
#@markdown **Stop the VTOE server first** (interrupt cell 5) to free VRAM.

import subprocess, sys, pathlib, time
import torch

# Stop VTOE server if running
try:
    subprocess.run(['pkill', '-f', 'vtoe_server'], timeout=5)
    import gc; gc.collect()
    torch.cuda.empty_cache()
    print('VTOE server stopped, VRAM freed')
except Exception:
    pass

subtypes = ['saree', 'lehenga', 'kurta']
epochs = 15
max_samples = 500
output_dir = pathlib.Path('/content/trained_loras')
output_dir.mkdir(exist_ok=True)

for subtype in subtypes:
    data_dir = f'/content/data/{subtype}'
    print(f'\n{"="*60}')
    print(f'  Training {subtype.upper()} LoRA ({epochs} epochs, {max_samples} samples)')
    print(f'{"="*60}')

    # Step 1: Prepare data
    print(f'\n[1/2] Generating {max_samples} synthetic training pairs...')
    result = subprocess.run(
        [sys.executable, '-m', 'vtoe.training.prepare_data',
         '--subtype', subtype, '--output', data_dir, '--max-samples', str(max_samples)],
        capture_output=True, text=True, timeout=600
    )
    if result.returncode != 0:
        print(f'Data prep failed: {result.stderr[:500]}')
        continue
    print(f'Data ready at {data_dir}')

    # Step 2: Train LoRA
    print(f'\n[2/2] Training LoRA for {epochs} epochs...')
    lora_path = output_dir / f'lora-{subtype}'
    result = subprocess.run(
        [sys.executable, '-m', 'vtoe.training.train_lora',
         '--subtype', subtype, '--data', data_dir,
         '--epochs', str(epochs), '--output', str(lora_path)],
        capture_output=True, text=True, timeout=7200
    )
    if result.returncode != 0:
        print(f'Training failed: {result.stderr[:500]}')
        continue
    print(f'LoRA saved to {lora_path}')
    print(result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)

print(f'\n\nAll LoRAs trained! Check {output_dir}')
print('Next: Upload to HuggingFace and update lora_registry.py')

In [ ]:
#@title 8. Upload trained LoRAs to HuggingFace
#@markdown Run after training completes. Requires HuggingFace token.

import subprocess, sys, pathlib, os

#@markdown Your HuggingFace username:
HF_USERNAME = 'Shuttler14'  #@param {type:'string'}

#@markdown HuggingFace token (from huggingface.co/settings/tokens):
HF_TOKEN = ''  #@param {type:'string'}

if not HF_TOKEN:
    print('Set HF_TOKEN first! Get token from huggingface.co/settings/tokens')
else:
    loras = list(pathlib.Path('/content/trained_loras').glob('lora-*'))
    if not loras:
        print('No trained LoRAs found. Run cell 7 first.')
    else:
        for lora_dir in loras:
            subtype = lora_dir.name.replace('lora-', '')
            repo_name = f'drishti-lora-{subtype}'
            print(f'\nUploading {repo_name}...')

            # Create repo if needed
            subprocess.run(
                ['huggingface-cli', 'repo', 'create', repo_name, '--type', 'model', '--exist-ok'],
                env={**os.environ, 'HF_TOKEN': HF_TOKEN}
            )

            # Upload files
            result = subprocess.run(
                ['huggingface-cli', 'upload', f'{HF_USERNAME}/{repo_name}',
                 str(lora_dir), '--include', '*.safetensors,*.json,*.txt'],
                env={**os.environ, 'HF_TOKEN': HF_TOKEN},
                capture_output=True, text=True
            )
            if result.returncode == 0:
                print(f'  Uploaded to https://huggingface.co/{HF_USERNAME}/{repo_name}')
            else:
                print(f'  Upload failed: {result.stderr[:300]}')

        print(f'\nDone! Update vtoe/ethnic/lora_registry.py with new repo URLs')

In [ ]:
#@title 9. Keep alive (prevents Colab idle disconnect)
import time, torch
print('Running keep-alive loop (Ctrl+M I to interrupt)...')
while True:
    _ = torch.zeros(1).cuda()  # keeps GPU active
    time.sleep(60)
    print(f'Alive at {time.strftime("%H:%M:%S")}')